# RSNA ATD — Organ Branch Pipeline v12 (balanced-selection, resumable, calibration-ready)

Main fixes in this version:
- `num_workers=0` to avoid Kaggle multiprocessing deadlocks.
- Five-minute fail-fast preflight for EfficientNet-B0 pretrained weights.
- Optional local backbone checkpoint for fully offline Kaggle runs.
- One cached backbone state reused by every training run; no repeated Hub calls.
- Atomic, combination-level checkpointing for all expensive stages.
- Unique model/prediction files for every organ/seed/hyperparameter run.
- Validation logits and probabilities saved for Phase 2 stacking/calibration.
- Final tuned multi-seed confirmation for kidney and spleen.

By default, only the still-outstanding stages run:
1. tuned five-seed confirmation for kidney and spleen at alpha=1.0, boost=1.0

The completed spleen alpha 3.0–6.0 sweep is retained as an optional audit stage,
but those configurations are rejected because class-1 recall collapsed to zero.

Enable earlier stages in RUN_STAGES only when they genuinely need recomputation.

In [1]:
# Install only when missing. This avoids an unnecessary reinstall on every rerun.
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "timm": "timm",
    "huggingface_hub": "huggingface_hub",
    "sklearn": "scikit-learn",
}

for module_name, pip_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

In [2]:
import ast
import gc
import json
import os
import random
import shutil
import time
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# -----------------------------
# Paths and core configuration
# -----------------------------
PHASE0_DIR = Path(
    "/kaggle/input/datasets/<username>/"
    "rsna-atd-phase0-preprocessed/processed"
)
MANIFEST_PATH = PHASE0_DIR / "manifest.csv"
LABELS_PATH = PHASE0_DIR / "subset_labels.csv"

OUTPUT_DIR = Path("/kaggle/working/atd_organ_pipeline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = OUTPUT_DIR / "models"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
CACHE_DIR = OUTPUT_DIR / "hf_home"

for directory in (MODEL_DIR, PREDICTION_DIR, CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = 256
N_SLICES_PER_ORGAN = 32
BATCH_SIZE = 4
EPOCHS = 5
LEARNING_RATE = 1e-4

SEEDS = [42, 123, 2024, 7, 555]
SWEEP_SEED = 42

ORGAN_NUM_CLASSES = {
    "liver": 3,
    "spleen": 3,
    "kidney": 3,
    "bowel": 2,
}

# Keep False for methodological consistency with the ordinal experiments
# already summarized: the old ordinal loop did not multiply by dataset sample weights.
ORDINAL_USE_SAMPLE_WEIGHT = False

USE_AMP = True
USE_DATA_PARALLEL = False  # Single GPU is more reproducible; set True only if needed.

# Set this to a Kaggle input checkpoint to avoid all network access, for example:
# LOCAL_BACKBONE_STATE_PATH = Path(
#     "/kaggle/input/your-backbone-dataset/efficientnet_b0_gray_backbone.pt"
# )
LOCAL_BACKBONE_STATE_PATH = None

BACKBONE_NAME = "efficientnet_b0"
BACKBONE_STATE_PATH = OUTPUT_DIR / "efficientnet_b0_gray_backbone.pt"
PRETRAINED_TIMEOUT_SECONDS = 300

# Run only the outstanding work by default.
RUN_STAGES = {
    "single_seed_baseline": False,
    "multiseed_baseline": False,
    "default_ordinal_multiseed": False,
    "initial_alpha_boost_sweep": False,
    "extended_alpha_sweep": False,
    "spleen_further_sweep": False,
    "tuned_multiseed_confirmation": True,
}

print("Output directory:", OUTPUT_DIR)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible CUDA devices:", torch.cuda.device_count())

Output directory: /kaggle/working/atd_organ_pipeline
PyTorch version: 2.10.0+cu128
CUDA available: True
Visible CUDA devices: 2


In [3]:
# -----------------------------
# Small utility functions
# -----------------------------
def atomic_write_csv(df: pd.DataFrame, path: Path) -> None:
    """Write a CSV atomically so an interruption does not leave a half-written file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(temp_path, index=False)
    temp_path.replace(path)


def read_results(path: Path, columns: list[str]) -> pd.DataFrame:
    if path.exists():
        df = pd.read_csv(path)
        for column in columns:
            if column not in df.columns:
                df[column] = np.nan
        return df
    return pd.DataFrame(columns=columns)


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Determinism helps make repeated alpha=3.0 checks comparable.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def float_key(value: float) -> float:
    return round(float(value), 8)


def float_tag(value: float) -> str:
    text = f"{float(value):g}"
    return text.replace("-", "m").replace(".", "p")


def make_run_tag(
    prefix: str,
    organ: str,
    seed: int,
    ordinal_alpha: float | None = None,
    present_boost: float | None = None,
) -> str:
    tag = f"{prefix}_{organ}_seed{seed}"
    if ordinal_alpha is not None:
        tag += f"_alpha{float_tag(ordinal_alpha)}"
    if present_boost is not None:
        tag += f"_boost{float_tag(present_boost)}"
    return tag


def clear_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [4]:
# -----------------------------
# Verify paths and load data
# -----------------------------
print("PHASE0_DIR exists:", PHASE0_DIR.exists())
print("Manifest exists:", MANIFEST_PATH.exists())
print("Labels exists:", LABELS_PATH.exists())

if not PHASE0_DIR.exists():
    raise FileNotFoundError(f"Missing Phase 0 directory: {PHASE0_DIR}")
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing manifest: {MANIFEST_PATH}")
if not LABELS_PATH.exists():
    raise FileNotFoundError(f"Missing labels: {LABELS_PATH}")

manifest_df = pd.read_csv(MANIFEST_PATH)
labels_df = pd.read_csv(LABELS_PATH)

manifest_df["output_path"] = manifest_df["output_path"].apply(
    lambda path: str(PHASE0_DIR / Path(path).name)
)
manifest_df["organs_cropped"] = manifest_df["organs_cropped"].apply(ast.literal_eval)
manifest_df["resize_factors"] = manifest_df["resize_factors"].apply(ast.literal_eval)

sample_path = Path(manifest_df.iloc[0]["output_path"])
print("Rewritten sample path:", sample_path)
print("Sample exists:", sample_path.exists())
print("Total series:", len(manifest_df))
print(
    "Series with organ crops:",
    int((manifest_df["organs_cropped"].apply(len) > 0).sum()),
)

PHASE0_DIR exists: True
Manifest exists: True
Labels exists: True
Rewritten sample path: /kaggle/input/datasets/<username>/rsna-atd-phase0-preprocessed/processed/10004_21057.npz
Sample exists: True
Total series: 1803
Series with organ crops: 206


In [5]:
# -----------------------------
# Recreate the fixed patient-level split
# -----------------------------
strat_df = labels_df[
    [
        "patient_id",
        "any_injury",
        "bowel_injury",
        "extravasation_injury",
        "kidney_high",
        "liver_high",
        "spleen_high",
    ]
].copy()

strat_df["strat_key"] = (
    strat_df["bowel_injury"].astype(str)
    + "_"
    + strat_df["extravasation_injury"].astype(str)
    + "_"
    + (
        strat_df[["kidney_high", "liver_high", "spleen_high"]].sum(axis=1) > 0
    )
    .astype(int)
    .astype(str)
)

bucket_counts = strat_df["strat_key"].value_counts()
rare_buckets = bucket_counts[bucket_counts < 2].index
strat_df.loc[strat_df["strat_key"].isin(rare_buckets), "strat_key"] = "other"

train_patient_series, val_patient_series = train_test_split(
    strat_df["patient_id"],
    test_size=0.15,
    random_state=42,
    stratify=strat_df["strat_key"],
)

train_patients = set(train_patient_series)
val_patients = set(val_patient_series)

print(
    f"Train patients: {len(train_patients)}, "
    f"Val patients: {len(val_patients)}"
)

manifest_labeled = manifest_df.merge(
    labels_df,
    on="patient_id",
    how="left",
    validate="many_to_one",
)

manifest_labeled["split"] = manifest_labeled["patient_id"].apply(
    lambda patient_id: (
        "train"
        if patient_id in train_patients
        else "val"
        if patient_id in val_patients
        else "unused"
    )
)

organ_manifest = manifest_labeled[
    manifest_labeled["organs_cropped"].apply(len) > 0
].reset_index(drop=True)

print("\nSeries usable for organ-branch training:", len(organ_manifest))
print(organ_manifest["split"].value_counts())
print("\nAny-injury mean inside organ-crop subset:")
print(organ_manifest.groupby("split")["any_injury"].mean())

atomic_write_csv(
    organ_manifest,
    OUTPUT_DIR / "organ_manifest_split.csv",
)

Train patients: 1020, Val patients: 180

Series usable for organ-branch training: 206
split
train    173
val       33
Name: count, dtype: int64

Any-injury mean inside organ-crop subset:
split
train    1.0
val      1.0
Name: any_injury, dtype: float64


In [6]:
# -----------------------------
# Dataset sanity report
# -----------------------------
def get_organ_label(row: pd.Series, organ: str) -> int:
    if organ == "bowel":
        return int(row["bowel_injury"])

    labels = [
        row[f"{organ}_healthy"],
        row[f"{organ}_low"],
        row[f"{organ}_high"],
    ]
    return int(np.argmax(labels))


for organ in ORGAN_NUM_CLASSES:
    available = organ_manifest[
        organ_manifest["organs_cropped"].apply(lambda organs: organ in organs)
    ].copy()
    available["label"] = available.apply(
        lambda row: get_organ_label(row, organ),
        axis=1,
    )

    print(f"\n=== {organ} (n={len(available)}) ===")
    print("Overall label counts:")
    print(available["label"].value_counts().sort_index())

    for split in ("train", "val"):
        split_df = available[available["split"] == split]
        print(f"\n{split.title()} split (n={len(split_df)}) label counts:")
        print(split_df["label"].value_counts().sort_index())


=== liver (n=206) ===
Overall label counts:
label
0    83
1    92
2    31
Name: count, dtype: int64

Train split (n=173) label counts:
label
0    69
1    77
2    27
Name: count, dtype: int64

Val split (n=33) label counts:
label
0    14
1    15
2     4
Name: count, dtype: int64

=== spleen (n=206) ===
Overall label counts:
label
0    67
1    59
2    80
Name: count, dtype: int64

Train split (n=173) label counts:
label
0    54
1    52
2    67
Name: count, dtype: int64

Val split (n=33) label counts:
label
0    13
1     7
2    13
Name: count, dtype: int64

=== kidney (n=204) ===
Overall label counts:
label
0    101
1     56
2     47
Name: count, dtype: int64

Train split (n=171) label counts:
label
0    86
1    49
2    36
Name: count, dtype: int64

Val split (n=33) label counts:
label
0    15
1     7
2    11
Name: count, dtype: int64

=== bowel (n=206) ===
Overall label counts:
label
0    184
1     22
Name: count, dtype: int64

Train split (n=173) label counts:
label
0    152
1     21
N

In [7]:
# -----------------------------
# Fail-fast pretrained-backbone preparation
# -----------------------------
#
# The previous run stalled before epoch 1 while `timm.create_model(...,
# pretrained=True)` attempted to retrieve weights. This cell either:
# 1. copies a user-supplied local backbone state;
# 2. reuses a completed state in OUTPUT_DIR; or
# 3. downloads and serializes the state in an isolated process with a hard timeout.
#
def prepare_backbone_state() -> Path:
    if LOCAL_BACKBONE_STATE_PATH is not None:
        local_path = Path(LOCAL_BACKBONE_STATE_PATH)
        if not local_path.exists():
            raise FileNotFoundError(
                f"LOCAL_BACKBONE_STATE_PATH does not exist: {local_path}"
            )
        shutil.copy2(local_path, BACKBONE_STATE_PATH)
        print("Copied local backbone state to:", BACKBONE_STATE_PATH)
        return BACKBONE_STATE_PATH

    if BACKBONE_STATE_PATH.exists():
        print("Reusing prepared backbone state:", BACKBONE_STATE_PATH)
        return BACKBONE_STATE_PATH

    env = os.environ.copy()
    env.update(
        {
            "HF_HOME": str(CACHE_DIR),
            "HF_HUB_CACHE": str(CACHE_DIR / "hub"),
            "HF_HUB_ETAG_TIMEOUT": "15",
            "HF_HUB_DOWNLOAD_TIMEOUT": "60",
            "HF_HUB_DISABLE_XET": "1",
            "HF_HUB_VERBOSITY": "info",
        }
    )

    probe_code = f"""
import time
from pathlib import Path

import timm
import torch

output_path = Path({str(BACKBONE_STATE_PATH)!r})
start = time.time()

print("Preparing pretrained {BACKBONE_NAME} with in_chans=1...", flush=True)
model = timm.create_model(
    {BACKBONE_NAME!r},
    pretrained=True,
    num_classes=0,
    in_chans=1,
)

payload = {{
    "model_name": {BACKBONE_NAME!r},
    "in_chans": 1,
    "num_features": int(model.num_features),
    "state_dict": {{
        key: value.detach().cpu()
        for key, value in model.state_dict().items()
    }},
}}

torch.save(payload, output_path)
print(
    f"Backbone prepared in {{time.time() - start:.1f}} seconds: {{output_path}}",
    flush=True,
)
"""

    print(
        "No local backbone state found. Starting isolated pretrained-weight "
        f"preflight with a {PRETRAINED_TIMEOUT_SECONDS}-second timeout."
    )

    try:
        completed = subprocess.run(
            [sys.executable, "-c", probe_code],
            env=env,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            timeout=PRETRAINED_TIMEOUT_SECONDS,
            check=True,
        )
        print(completed.stdout)

    except subprocess.TimeoutExpired as exc:
        partial_output = exc.stdout or ""
        raise RuntimeError(
            "Pretrained backbone preparation exceeded the timeout. "
            "Training was intentionally not started. Attach a local backbone "
            "state as a Kaggle dataset and set LOCAL_BACKBONE_STATE_PATH.\n\n"
            f"Partial output:\n{partial_output}"
        ) from exc

    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Pretrained backbone preparation failed. Training was intentionally "
            "not started. Attach a local backbone state as a Kaggle dataset and "
            "set LOCAL_BACKBONE_STATE_PATH.\n\n"
            f"Output:\n{exc.stdout}"
        ) from exc

    if not BACKBONE_STATE_PATH.exists():
        raise RuntimeError(
            "The preflight process returned without creating the backbone state."
        )

    return BACKBONE_STATE_PATH


prepared_backbone_path = prepare_backbone_state()
print("Prepared backbone path:", prepared_backbone_path)

No local backbone state found. Starting isolated pretrained-weight preflight with a 300-second timeout.
Preparing pretrained efficientnet_b0 with in_chans=1...
Backbone prepared in 0.9 seconds: /kaggle/working/atd_organ_pipeline/efficientnet_b0_gray_backbone.pt

Prepared backbone path: /kaggle/working/atd_organ_pipeline/efficientnet_b0_gray_backbone.pt


In [8]:
# -----------------------------
# Import timm offline and load one reusable CPU template
# -----------------------------
os.environ.update(
    {
        "HF_HOME": str(CACHE_DIR),
        "HF_HUB_CACHE": str(CACHE_DIR / "hub"),
        "HF_HUB_OFFLINE": "1",
        "HF_HUB_DISABLE_XET": "1",
    }
)

import timm

backbone_payload = torch.load(
    prepared_backbone_path,
    map_location="cpu",
)

if "state_dict" not in backbone_payload:
    raise ValueError(
        "Backbone file must contain a dictionary with a 'state_dict' key."
    )

PRETRAINED_BACKBONE_STATE = backbone_payload["state_dict"]
PRETRAINED_FEATURE_DIM = int(backbone_payload["num_features"])

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
amp_enabled = bool(USE_AMP and device.type == "cuda")

print("timm version:", timm.__version__)
print("Using device:", device)
print("AMP enabled:", amp_enabled)

timm version: 1.0.26
Using device: cuda:0
AMP enabled: True


In [9]:
# -----------------------------
# Dataset and fixed DataLoader
# -----------------------------
class OrganCropDataset(Dataset):
    def __init__(
        self,
        manifest: pd.DataFrame,
        organ: str,
        n_slices: int = N_SLICES_PER_ORGAN,
        img_size: int = TARGET_SIZE,
    ):
        self.df = manifest[
            manifest["organs_cropped"].apply(lambda organs: organ in organs)
        ].reset_index(drop=True)
        self.organ = organ
        self.n_slices = n_slices
        self.img_size = img_size

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, index: int):
        row = self.df.iloc[index]
        archive = np.load(row["output_path"])

        if self.organ not in archive.files:
            raise KeyError(
                f"{self.organ!r} missing from archive {row['output_path']}"
            )

        volume = archive[self.organ].astype(np.float32) / 255.0

        if volume.ndim != 3:
            raise ValueError(
                f"Expected [slices, height, width] for {self.organ}; "
                f"got shape {volume.shape} in {row['output_path']}"
            )

        label = get_organ_label(row, self.organ)
        sample_weight = self._get_sample_weight(row)

        volume_tensor = torch.from_numpy(volume).unsqueeze(1)

        return {
            "volume": volume_tensor,
            "label": torch.tensor(label, dtype=torch.long),
            "sample_weight": torch.tensor(sample_weight, dtype=torch.float32),
            "patient_id": str(row["patient_id"]),
            "series_id": str(row["series_id"]),
        }

    def _get_sample_weight(self, row: pd.Series) -> float:
        if self.organ == "bowel":
            return 2.0 if int(row["bowel_injury"]) == 1 else 1.0

        if int(row[f"{self.organ}_high"]) == 1:
            return 4.0
        if int(row[f"{self.organ}_low"]) == 1:
            return 2.0
        return 1.0


def make_loader(
    manifest: pd.DataFrame,
    organ: str,
    split: str,
    batch_size: int = BATCH_SIZE,
    num_workers: int = 0,
) -> DataLoader:
    split_df = manifest[manifest["split"] == split]
    dataset = OrganCropDataset(split_df, organ)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=(split == "train"),
        num_workers=num_workers,  # Critical Kaggle deadlock fix.
        pin_memory=(device.type == "cuda"),
        drop_last=False,
    )

In [10]:
# -----------------------------
# Model and losses
# -----------------------------
class SliceAttentionPool(nn.Module):
    def __init__(self, feature_dim: int):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        weights = torch.softmax(self.attention(features), dim=1)
        return (features * weights).sum(dim=1)


class OrganClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        backbone_name: str = BACKBONE_NAME,
    ):
        super().__init__()

        # Architecture creation is strictly offline.
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=False,
            num_classes=0,
            in_chans=1,
        )
        self.backbone.load_state_dict(
            PRETRAINED_BACKBONE_STATE,
            strict=True,
        )

        feature_dim = int(self.backbone.num_features)
        if feature_dim != PRETRAINED_FEATURE_DIM:
            raise ValueError(
                f"Backbone feature mismatch: {feature_dim} vs "
                f"{PRETRAINED_FEATURE_DIM}"
            )

        self.pool = SliceAttentionPool(feature_dim)
        self.head = nn.Linear(feature_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, slices, channels, height, width = x.shape

        x = x.reshape(
            batch_size * slices,
            channels,
            height,
            width,
        )

        features = self.backbone(x).reshape(
            batch_size,
            slices,
            -1,
        )

        pooled = self.pool(features)
        return self.head(pooled)


class OrdinalPresentLabelUpweightingLoss(nn.Module):
    def __init__(
        self,
        class_counts: list[int],
        beta: float = 0.999,
        ordinal_alpha: float = 0.5,
        present_boost: float = 1.5,
        focal_gamma: float = 1.0,
    ):
        super().__init__()

        counts = np.asarray(class_counts, dtype=np.float32)
        effective_number = 1.0 - np.power(beta, counts)
        weights = (1.0 - beta) / np.clip(
            effective_number,
            1e-8,
            None,
        )
        weights = weights / weights.sum() * len(counts)

        self.register_buffer(
            "class_weights",
            torch.tensor(weights, dtype=torch.float32),
        )

        self.ordinal_alpha = float(ordinal_alpha)
        self.present_boost = float(present_boost)
        self.focal_gamma = float(focal_gamma)
        self.num_classes = len(class_counts)

    def forward(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor,
        sample_weight: torch.Tensor | None = None,
    ) -> torch.Tensor:
        probabilities = torch.softmax(logits, dim=1)
        log_probabilities = torch.log_softmax(logits, dim=1)

        target_probabilities = probabilities.gather(
            1,
            targets.unsqueeze(1),
        ).squeeze(1)

        focal_factor = (
            1.0 - target_probabilities
        ) ** self.focal_gamma

        cross_entropy = -log_probabilities.gather(
            1,
            targets.unsqueeze(1),
        ).squeeze(1)

        class_weight = self.class_weights[targets]
        base_loss = focal_factor * class_weight * cross_entropy

        class_ranks = torch.arange(
            self.num_classes,
            device=logits.device,
            dtype=probabilities.dtype,
        )

        expected_rank = (
            probabilities * class_ranks.unsqueeze(0)
        ).sum(dim=1)

        ordinal_penalty = (
            expected_rank - targets.float()
        ) ** 2

        present_mask = (targets > 0).float()
        present_weight = (
            1.0
            + (self.present_boost - 1.0) * present_mask
        )

        loss = present_weight * (
            base_loss
            + self.ordinal_alpha * ordinal_penalty
        )

        if sample_weight is not None:
            loss = loss * sample_weight

        return loss.mean()


def get_class_counts(
    manifest: pd.DataFrame,
    organ: str,
    split: str = "train",
) -> list[int]:
    subset = manifest[
        manifest["organs_cropped"].apply(lambda organs: organ in organs)
        & (manifest["split"] == split)
    ]

    labels = subset.apply(
        lambda row: get_organ_label(row, organ),
        axis=1,
    )

    return [
        int(np.sum(labels == class_index))
        for class_index in range(ORGAN_NUM_CLASSES[organ])
    ]

In [11]:
# -----------------------------
# Unified train/evaluate function
# -----------------------------
RESULT_COLUMNS = [
    "run_tag",
    "loss_mode",
    "organ",
    "seed",
    "ordinal_alpha",
    "present_boost",
    "focal_gamma",
    "best_epoch",
    "val_loss",
    "val_acc",
    "recall_class0",
    "recall_class1",
    "recall_class2",
    "macro_recall",
    "checkpoint_path",
    "prediction_path",
]


def train_and_evaluate(
    organ: str,
    loss_mode: str,
    run_tag: str,
    epochs: int = EPOCHS,
    learning_rate: float = LEARNING_RATE,
    seed: int = 42,
    ordinal_alpha: float = 0.5,
    present_boost: float = 1.5,
    focal_gamma: float = 1.0,
) -> dict:
    if loss_mode not in {"baseline", "ordinal"}:
        raise ValueError("loss_mode must be 'baseline' or 'ordinal'")

    set_seed(seed)

    train_loader = make_loader(
        organ_manifest,
        organ,
        "train",
    )
    val_loader = make_loader(
        organ_manifest,
        organ,
        "val",
    )

    print(
        f"[{run_tag}] train={len(train_loader.dataset)}, "
        f"val={len(val_loader.dataset)}"
    )

    base_model = OrganClassifier(
        num_classes=ORGAN_NUM_CLASSES[organ]
    ).to(device)

    if USE_DATA_PARALLEL and torch.cuda.device_count() > 1:
        model = nn.DataParallel(base_model)
    else:
        model = base_model

    optimizer = optim.AdamW(
        model.parameters(),
        lr=learning_rate,
    )

    if loss_mode == "baseline":
        criterion = nn.CrossEntropyLoss(reduction="none")
    else:
        class_counts = get_class_counts(
            organ_manifest,
            organ,
            split="train",
        )
        print(f"[{run_tag}] class_counts={class_counts}")

        criterion = OrdinalPresentLabelUpweightingLoss(
            class_counts=class_counts,
            ordinal_alpha=ordinal_alpha,
            present_boost=present_boost,
            focal_gamma=focal_gamma,
        ).to(device)

    scaler = torch.cuda.amp.GradScaler(
        enabled=amp_enabled
    )

    checkpoint_path = MODEL_DIR / f"{run_tag}.pt"
    prediction_path = PREDICTION_DIR / f"{run_tag}.csv"
    history_path = OUTPUT_DIR / f"{run_tag}_history.csv"

    best_val_loss = float("inf")
    best_epoch = None
    best_prediction_frame = None
    history_rows = []

    for epoch_index in range(epochs):
        epoch_number = epoch_index + 1

        model.train()
        training_losses = []

        progress = tqdm(
            train_loader,
            desc=f"{run_tag} epoch {epoch_number}/{epochs}",
            leave=False,
        )

        for batch in progress:
            volumes = batch["volume"].to(
                device,
                non_blocking=True,
            )
            labels = batch["label"].to(
                device,
                non_blocking=True,
            )
            sample_weights = batch["sample_weight"].to(
                device,
                non_blocking=True,
            )

            optimizer.zero_grad(set_to_none=True)

            autocast_context = (
                torch.cuda.amp.autocast(enabled=True)
                if amp_enabled
                else nullcontext()
            )

            with autocast_context:
                logits = model(volumes)

                if loss_mode == "baseline":
                    per_sample_loss = criterion(
                        logits,
                        labels,
                    )
                    loss = (
                        per_sample_loss * sample_weights
                    ).mean()
                else:
                    ordinal_sample_weight = (
                        sample_weights
                        if ORDINAL_USE_SAMPLE_WEIGHT
                        else None
                    )
                    loss = criterion(
                        logits,
                        labels,
                        sample_weight=ordinal_sample_weight,
                    )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            training_losses.append(float(loss.detach().cpu()))
            progress.set_postfix(loss=f"{training_losses[-1]:.4f}")

        model.eval()

        validation_losses = []
        validation_rows = []

        with torch.no_grad():
            for batch in val_loader:
                volumes = batch["volume"].to(
                    device,
                    non_blocking=True,
                )
                labels = batch["label"].to(
                    device,
                    non_blocking=True,
                )
                sample_weights = batch["sample_weight"].to(
                    device,
                    non_blocking=True,
                )

                autocast_context = (
                    torch.cuda.amp.autocast(enabled=True)
                    if amp_enabled
                    else nullcontext()
                )

                with autocast_context:
                    logits = model(volumes)

                    if loss_mode == "baseline":
                        per_sample_loss = criterion(
                            logits,
                            labels,
                        )
                        loss = (
                            per_sample_loss * sample_weights
                        ).mean()
                    else:
                        ordinal_sample_weight = (
                            sample_weights
                            if ORDINAL_USE_SAMPLE_WEIGHT
                            else None
                        )
                        loss = criterion(
                            logits,
                            labels,
                            sample_weight=ordinal_sample_weight,
                        )

                probabilities = torch.softmax(
                    logits.float(),
                    dim=1,
                )
                predictions = torch.argmax(
                    probabilities,
                    dim=1,
                )

                validation_losses.append(
                    float(loss.detach().cpu())
                )

                logits_cpu = logits.float().cpu().numpy()
                probabilities_cpu = probabilities.cpu().numpy()
                labels_cpu = labels.cpu().numpy()
                predictions_cpu = predictions.cpu().numpy()

                for row_index in range(len(labels_cpu)):
                    row = {
                        "patient_id": batch["patient_id"][row_index],
                        "series_id": batch["series_id"][row_index],
                        "true_label": int(labels_cpu[row_index]),
                        "pred_label": int(predictions_cpu[row_index]),
                    }

                    for class_index in range(
                        ORGAN_NUM_CLASSES[organ]
                    ):
                        row[f"logit_class{class_index}"] = float(
                            logits_cpu[row_index, class_index]
                        )
                        row[f"prob_class{class_index}"] = float(
                            probabilities_cpu[row_index, class_index]
                        )

                    validation_rows.append(row)

        train_loss = (
            float(np.mean(training_losses))
            if training_losses
            else float("nan")
        )
        val_loss = (
            float(np.mean(validation_losses))
            if validation_losses
            else float("nan")
        )

        validation_frame = pd.DataFrame(validation_rows)
        val_acc = float(
            (
                validation_frame["true_label"]
                == validation_frame["pred_label"]
            ).mean()
        )

        print(
            f"[{run_tag}] epoch={epoch_number} "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_loss:.4f} "
            f"val_acc={val_acc:.3f}"
        )

        history_rows.append(
            {
                "run_tag": run_tag,
                "organ": organ,
                "loss_mode": loss_mode,
                "seed": seed,
                "epoch": epoch_number,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_acc": val_acc,
            }
        )
        atomic_write_csv(
            pd.DataFrame(history_rows),
            history_path,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch_number
            best_prediction_frame = validation_frame.copy()

            state_dict = (
                model.module.state_dict()
                if isinstance(model, nn.DataParallel)
                else model.state_dict()
            )

            temporary_checkpoint = checkpoint_path.with_suffix(".pt.tmp")
            torch.save(
                {
                    "run_tag": run_tag,
                    "organ": organ,
                    "loss_mode": loss_mode,
                    "seed": seed,
                    "ordinal_alpha": ordinal_alpha,
                    "present_boost": present_boost,
                    "focal_gamma": focal_gamma,
                    "best_epoch": best_epoch,
                    "best_val_loss": best_val_loss,
                    "model_state_dict": {
                        key: value.detach().cpu()
                        for key, value in state_dict.items()
                    },
                },
                temporary_checkpoint,
            )
            temporary_checkpoint.replace(checkpoint_path)

            atomic_write_csv(
                best_prediction_frame,
                prediction_path,
            )

    if best_prediction_frame is None or best_epoch is None:
        raise RuntimeError(f"No validation predictions saved for {run_tag}")

    report = classification_report(
        best_prediction_frame["true_label"],
        best_prediction_frame["pred_label"],
        labels=list(range(ORGAN_NUM_CLASSES[organ])),
        output_dict=True,
        zero_division=0,
    )

    recalls = {
        class_index: float(
            report[str(class_index)]["recall"]
        )
        for class_index in range(
            ORGAN_NUM_CLASSES[organ]
        )
    }

    macro_recall = float(np.mean(list(recalls.values())))

    result = {
        "run_tag": run_tag,
        "loss_mode": loss_mode,
        "organ": organ,
        "seed": int(seed),
        "ordinal_alpha": (
            float(ordinal_alpha)
            if loss_mode == "ordinal"
            else np.nan
        ),
        "present_boost": (
            float(present_boost)
            if loss_mode == "ordinal"
            else np.nan
        ),
        "focal_gamma": (
            float(focal_gamma)
            if loss_mode == "ordinal"
            else np.nan
        ),
        "best_epoch": int(best_epoch),
        "val_loss": float(best_val_loss),
        "val_acc": float(
            (
                best_prediction_frame["true_label"]
                == best_prediction_frame["pred_label"]
            ).mean()
        ),
        "recall_class0": recalls.get(0, np.nan),
        "recall_class1": recalls.get(1, np.nan),
        "recall_class2": recalls.get(2, np.nan),
        "macro_recall": macro_recall,
        "checkpoint_path": str(checkpoint_path),
        "prediction_path": str(prediction_path),
    }

    del model
    del base_model
    del optimizer
    clear_memory()

    return result

In [12]:
# -----------------------------
# Stage 1: optional single-seed baseline
# -----------------------------
baseline_single_path = OUTPUT_DIR / "baseline_single_seed_results.csv"

if RUN_STAGES["single_seed_baseline"]:
    baseline_single_df = read_results(
        baseline_single_path,
        RESULT_COLUMNS,
    )
    completed_organs = set(
        baseline_single_df["organ"].dropna().astype(str)
    )

    rows = baseline_single_df.to_dict("records")

    for organ in ["liver", "spleen", "kidney", "bowel"]:
        if organ in completed_organs:
            print(f"[baseline single] {organ} already completed.")
            continue

        run_tag = make_run_tag(
            prefix="baseline_single",
            organ=organ,
            seed=42,
        )

        result = train_and_evaluate(
            organ=organ,
            loss_mode="baseline",
            run_tag=run_tag,
            seed=42,
        )
        rows.append(result)
        atomic_write_csv(
            pd.DataFrame(rows),
            baseline_single_path,
        )

    baseline_single_df = pd.DataFrame(rows)
    print(baseline_single_df.to_string(index=False))
else:
    print("Skipping optional single-seed baseline stage.")

Skipping optional single-seed baseline stage.


In [13]:
# -----------------------------
# Stage 2: optional multi-seed baseline
# -----------------------------
baseline_multiseed_path = OUTPUT_DIR / "multiseed_baseline_results.csv"

if RUN_STAGES["multiseed_baseline"]:
    baseline_multiseed_df = read_results(
        baseline_multiseed_path,
        RESULT_COLUMNS,
    )

    completed = {
        (str(row.organ), int(row.seed))
        for row in baseline_multiseed_df.itertuples()
        if pd.notna(row.organ) and pd.notna(row.seed)
    }

    rows = baseline_multiseed_df.to_dict("records")

    for organ in ["liver", "spleen", "kidney"]:
        for seed in SEEDS:
            key = (organ, int(seed))
            if key in completed:
                print(f"[baseline multiseed] {key} already completed.")
                continue

            run_tag = make_run_tag(
                prefix="baseline_multiseed",
                organ=organ,
                seed=seed,
            )

            result = train_and_evaluate(
                organ=organ,
                loss_mode="baseline",
                run_tag=run_tag,
                seed=seed,
            )
            rows.append(result)
            atomic_write_csv(
                pd.DataFrame(rows),
                baseline_multiseed_path,
            )

    baseline_multiseed_df = pd.DataFrame(rows)
else:
    print("Skipping optional multi-seed baseline stage.")

Skipping optional multi-seed baseline stage.


In [14]:
# -----------------------------
# Stage 3: optional default ordinal multi-seed
# -----------------------------
default_ordinal_path = OUTPUT_DIR / "multiseed_ordinal_default_results.csv"

if RUN_STAGES["default_ordinal_multiseed"]:
    default_ordinal_df = read_results(
        default_ordinal_path,
        RESULT_COLUMNS,
    )

    completed = {
        (str(row.organ), int(row.seed))
        for row in default_ordinal_df.itertuples()
        if pd.notna(row.organ) and pd.notna(row.seed)
    }

    rows = default_ordinal_df.to_dict("records")

    for organ in ["liver", "spleen", "kidney"]:
        for seed in SEEDS:
            key = (organ, int(seed))
            if key in completed:
                print(f"[default ordinal] {key} already completed.")
                continue

            run_tag = make_run_tag(
                prefix="ordinal_default",
                organ=organ,
                seed=seed,
                ordinal_alpha=0.5,
                present_boost=1.5,
            )

            result = train_and_evaluate(
                organ=organ,
                loss_mode="ordinal",
                run_tag=run_tag,
                seed=seed,
                ordinal_alpha=0.5,
                present_boost=1.5,
            )
            rows.append(result)
            atomic_write_csv(
                pd.DataFrame(rows),
                default_ordinal_path,
            )

    default_ordinal_df = pd.DataFrame(rows)
else:
    print("Skipping optional default ordinal multi-seed stage.")

Skipping optional default ordinal multi-seed stage.


In [15]:
# -----------------------------
# Stage 4: optional initial alpha x boost sweep
# -----------------------------
ALPHA_GRID = [0.1, 0.5, 1.0]
BOOST_GRID = [1.0, 1.5, 2.0]
SWEEP_ORGANS = ["kidney", "spleen"]

initial_sweep_path = OUTPUT_DIR / "hyperparameter_sweep_results.csv"

if RUN_STAGES["initial_alpha_boost_sweep"]:
    initial_sweep_df = read_results(
        initial_sweep_path,
        RESULT_COLUMNS,
    )

    completed = {
        (
            str(row.organ),
            float_key(row.ordinal_alpha),
            float_key(row.present_boost),
        )
        for row in initial_sweep_df.itertuples()
        if (
            pd.notna(row.organ)
            and pd.notna(row.ordinal_alpha)
            and pd.notna(row.present_boost)
        )
    }

    rows = initial_sweep_df.to_dict("records")

    for organ in SWEEP_ORGANS:
        for alpha in ALPHA_GRID:
            for boost in BOOST_GRID:
                key = (
                    organ,
                    float_key(alpha),
                    float_key(boost),
                )
                if key in completed:
                    print(f"[initial sweep] {key} already completed.")
                    continue

                run_tag = make_run_tag(
                    prefix="initial_sweep",
                    organ=organ,
                    seed=SWEEP_SEED,
                    ordinal_alpha=alpha,
                    present_boost=boost,
                )

                result = train_and_evaluate(
                    organ=organ,
                    loss_mode="ordinal",
                    run_tag=run_tag,
                    seed=SWEEP_SEED,
                    ordinal_alpha=alpha,
                    present_boost=boost,
                )
                rows.append(result)
                atomic_write_csv(
                    pd.DataFrame(rows),
                    initial_sweep_path,
                )

    initial_sweep_df = pd.DataFrame(rows)
else:
    print("Skipping optional initial alpha x boost sweep.")

Skipping optional initial alpha x boost sweep.


In [16]:
# -----------------------------
# Stage 5: optional extended alpha sweep at boost=1.0
# -----------------------------
ALPHA_GRID_EXTENDED = [1.0, 1.5, 2.0, 3.0]
FIXED_BOOST = 1.0

extended_sweep_path = OUTPUT_DIR / "extended_alpha_sweep.csv"

if RUN_STAGES["extended_alpha_sweep"]:
    extended_sweep_df = read_results(
        extended_sweep_path,
        RESULT_COLUMNS,
    )

    completed = {
        (
            str(row.organ),
            float_key(row.ordinal_alpha),
        )
        for row in extended_sweep_df.itertuples()
        if pd.notna(row.organ) and pd.notna(row.ordinal_alpha)
    }

    rows = extended_sweep_df.to_dict("records")

    for organ in SWEEP_ORGANS:
        for alpha in ALPHA_GRID_EXTENDED:
            key = (organ, float_key(alpha))
            if key in completed:
                print(f"[extended sweep] {key} already completed.")
                continue

            run_tag = make_run_tag(
                prefix="extended_sweep",
                organ=organ,
                seed=SWEEP_SEED,
                ordinal_alpha=alpha,
                present_boost=FIXED_BOOST,
            )

            result = train_and_evaluate(
                organ=organ,
                loss_mode="ordinal",
                run_tag=run_tag,
                seed=SWEEP_SEED,
                ordinal_alpha=alpha,
                present_boost=FIXED_BOOST,
            )
            rows.append(result)
            atomic_write_csv(
                pd.DataFrame(rows),
                extended_sweep_path,
            )

    extended_sweep_df = pd.DataFrame(rows)
else:
    print("Skipping optional extended alpha sweep.")

Skipping optional extended alpha sweep.


In [17]:
# -----------------------------
# Stage 6: outstanding spleen alpha sweep 3.0–6.0
# -----------------------------
ALPHA_GRID_SPLEEN_FURTHER = [3.0, 4.0, 5.0, 6.0]
SPLEEN_FIXED_BOOST = 1.0

spleen_further_path = OUTPUT_DIR / "spleen_further_sweep.csv"

if RUN_STAGES["spleen_further_sweep"]:
    spleen_further_df = read_results(
        spleen_further_path,
        RESULT_COLUMNS,
    )

    completed_alphas = {
        float_key(value)
        for value in spleen_further_df[
            "ordinal_alpha"
        ].dropna()
    }

    rows = spleen_further_df.to_dict("records")

    for alpha in ALPHA_GRID_SPLEEN_FURTHER:
        if float_key(alpha) in completed_alphas:
            print(
                f"[spleen further] alpha={alpha} already completed."
            )
            continue

        run_tag = make_run_tag(
            prefix="spleen_further",
            organ="spleen",
            seed=SWEEP_SEED,
            ordinal_alpha=alpha,
            present_boost=SPLEEN_FIXED_BOOST,
        )

        result = train_and_evaluate(
            organ="spleen",
            loss_mode="ordinal",
            run_tag=run_tag,
            seed=SWEEP_SEED,
            ordinal_alpha=alpha,
            present_boost=SPLEEN_FIXED_BOOST,
        )

        rows.append(result)
        atomic_write_csv(
            pd.DataFrame(rows),
            spleen_further_path,
        )

    spleen_further_df = pd.DataFrame(rows).sort_values(
        ["ordinal_alpha", "present_boost"]
    )

    print("\nSpleen further-sweep results:")
    print(
        spleen_further_df[
            [
                "ordinal_alpha",
                "present_boost",
                "best_epoch",
                "val_loss",
                "val_acc",
                "recall_class0",
                "recall_class1",
                "recall_class2",
                "macro_recall",
            ]
        ].to_string(index=False)
    )
else:
    spleen_further_df = read_results(
        spleen_further_path,
        RESULT_COLUMNS,
    )
    print("Skipping spleen further sweep.")

Skipping spleen further sweep.


In [18]:
# -----------------------------
# Select final kidney and spleen configurations
# -----------------------------
#
# Final decision:
# - Kidney: alpha=1.0, boost=1.0 (confirmed optimum from the earlier sweep).
# - Spleen: alpha=1.0, boost=1.0 (best balanced single-seed result).
#
# The completed spleen alpha 3.0–6.0 sweep is not used for final selection:
# every high-alpha candidate had recall_class1=0.0. Accuracy-only selection
# would therefore choose a clinically degenerate model.
#
MIN_CLASS_RECALL = 0.20

BEST_CONFIGS = {
    "kidney": {
        "ordinal_alpha": 1.0,
        "present_boost": 1.0,
    },
    "spleen": {
        "ordinal_alpha": 1.0,
        "present_boost": 1.0,
    },
}

# Audit the completed high-alpha sweep when its CSV is available.
if len(spleen_further_df):
    audit_columns = [
        "ordinal_alpha",
        "present_boost",
        "val_acc",
        "recall_class0",
        "recall_class1",
        "recall_class2",
        "macro_recall",
    ]

    spleen_selection_audit = spleen_further_df.copy()
    spleen_selection_audit["passes_min_class_recall"] = (
        (spleen_selection_audit["recall_class0"] >= MIN_CLASS_RECALL)
        & (spleen_selection_audit["recall_class1"] >= MIN_CLASS_RECALL)
        & (spleen_selection_audit["recall_class2"] >= MIN_CLASS_RECALL)
    )

    print("\nHigh-alpha spleen selection audit:")
    print(
        spleen_selection_audit[
            audit_columns + ["passes_min_class_recall"]
        ].sort_values("ordinal_alpha").to_string(index=False)
    )

    rejected_high_alpha = spleen_selection_audit[
        ~spleen_selection_audit["passes_min_class_recall"]
    ]

    if len(rejected_high_alpha):
        print(
            "\nRejected high-alpha spleen configurations because at least "
            f"one class recall was below {MIN_CLASS_RECALL:.2f}."
        )

    atomic_write_csv(
        spleen_selection_audit,
        OUTPUT_DIR / "spleen_selection_audit.csv",
    )

print("\nLOCKED BEST_CONFIGS:")
print(json.dumps(BEST_CONFIGS, indent=2))

# Always overwrite the previous automatically selected config, including any
# stale alpha=5.0 selection produced by v11.
with open(
    OUTPUT_DIR / "best_configs.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(BEST_CONFIGS, file, indent=2)


LOCKED BEST_CONFIGS:
{
  "kidney": {
    "ordinal_alpha": 1.0,
    "present_boost": 1.0
  },
  "spleen": {
    "ordinal_alpha": 1.0,
    "present_boost": 1.0
  }
}


In [19]:
# -----------------------------
# Stage 7: tuned five-seed confirmation
# -----------------------------
#
# Use a configuration-specific result file so stale v11 alpha=5 spleen rows
# cannot contaminate the alpha=1 confirmation or its summary.
TUNED_CONFIG_SIGNATURE = "kidney_a1_b1__spleen_a1_b1"
tuned_confirmation_path = (
    OUTPUT_DIR
    / f"multiseed_tuned_{TUNED_CONFIG_SIGNATURE}_results.csv"
)

configs_are_complete = all(
    config["ordinal_alpha"] is not None
    and config["present_boost"] is not None
    for config in BEST_CONFIGS.values()
)

if (
    RUN_STAGES["tuned_multiseed_confirmation"]
    and configs_are_complete
):
    tuned_df = read_results(
        tuned_confirmation_path,
        RESULT_COLUMNS,
    )

    # Defensive filtering: retain only rows matching the currently locked
    # organ-specific configuration.
    if len(tuned_df):
        active_row_mask = pd.Series(False, index=tuned_df.index)

        for active_organ, active_config in BEST_CONFIGS.items():
            active_row_mask |= (
                (tuned_df["organ"].astype(str) == active_organ)
                & (
                    tuned_df["ordinal_alpha"].astype(float).round(8)
                    == float_key(active_config["ordinal_alpha"])
                )
                & (
                    tuned_df["present_boost"].astype(float).round(8)
                    == float_key(active_config["present_boost"])
                )
            )

        removed_rows = int((~active_row_mask).sum())
        if removed_rows:
            print(
                f"Ignoring {removed_rows} stale tuned-confirmation row(s) "
                "that do not match BEST_CONFIGS."
            )

        tuned_df = tuned_df[active_row_mask].reset_index(drop=True)

    completed = {
        (
            str(row.organ),
            int(row.seed),
            float_key(row.ordinal_alpha),
            float_key(row.present_boost),
        )
        for row in tuned_df.itertuples()
        if (
            pd.notna(row.organ)
            and pd.notna(row.seed)
            and pd.notna(row.ordinal_alpha)
            and pd.notna(row.present_boost)
        )
    }

    rows = tuned_df.to_dict("records")

    for organ in ["kidney", "spleen"]:
        config = BEST_CONFIGS[organ]
        alpha = float(config["ordinal_alpha"])
        boost = float(config["present_boost"])

        for seed in SEEDS:
            key = (
                organ,
                int(seed),
                float_key(alpha),
                float_key(boost),
            )

            if key in completed:
                print(f"[tuned confirmation] {key} already completed.")
                continue

            run_tag = make_run_tag(
                prefix="tuned_confirmation",
                organ=organ,
                seed=seed,
                ordinal_alpha=alpha,
                present_boost=boost,
            )

            result = train_and_evaluate(
                organ=organ,
                loss_mode="ordinal",
                run_tag=run_tag,
                seed=seed,
                ordinal_alpha=alpha,
                present_boost=boost,
            )

            rows.append(result)
            atomic_write_csv(
                pd.DataFrame(rows),
                tuned_confirmation_path,
            )

    tuned_df = pd.DataFrame(rows)

elif not configs_are_complete:
    tuned_df = read_results(
        tuned_confirmation_path,
        RESULT_COLUMNS,
    )
    print(
        "Tuned confirmation was not started because the spleen "
        "sweep is incomplete."
    )
else:
    tuned_df = read_results(
        tuned_confirmation_path,
        RESULT_COLUMNS,
    )
    print("Skipping tuned multi-seed confirmation.")

[tuned_confirmation_kidney_seed42_alpha1_boost1] train=171, val=33
[tuned_confirmation_kidney_seed42_alpha1_boost1] class_counts=[86, 49, 36]


/tmp/ipykernel_23/2578973256.py:87: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(


tuned_confirmation_kidney_seed42_alpha1_boost1 epoch 1/5:   0%|          | 0/43 [00:00<?, ?it/s]

/tmp/ipykernel_23/2578973256.py:129: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  torch.cuda.amp.autocast(enabled=True)
/tmp/ipykernel_23/2578973256.py:185: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  torch.cuda.amp.autocast(enabled=True)


[tuned_confirmation_kidney_seed42_alpha1_boost1] epoch=1 train_loss=1.2927 val_loss=1.3811 val_acc=0.273


...

[tuned_confirmation_spleen_seed555_alpha1_boost1] epoch=3 train_loss=1.0950 val_loss=0.9143 val_acc=0.667


tuned_confirmation_spleen_seed555_alpha1_boost1 epoch 4/5:   0%|          | 0/44 [00:00<?, ?it/s]

[tuned_confirmation_spleen_seed555_alpha1_boost1] epoch=4 train_loss=0.9023 val_loss=0.9707 val_acc=0.697


tuned_confirmation_spleen_seed555_alpha1_boost1 epoch 5/5:   0%|          | 0/44 [00:00<?, ?it/s]

[tuned_confirmation_spleen_seed555_alpha1_boost1] epoch=5 train_loss=0.5762 val_loss=0.8931 val_acc=0.606


In [20]:
# -----------------------------
# Final tuned summary
# -----------------------------
if len(tuned_df):
    print("\nTUNED MULTI-SEED SUMMARY")

    summary_rows = []

    for organ in ["kidney", "spleen"]:
        active_config = BEST_CONFIGS[organ]
        subset = tuned_df[
            (tuned_df["organ"] == organ)
            & (
                tuned_df["ordinal_alpha"].astype(float).round(8)
                == float_key(active_config["ordinal_alpha"])
            )
            & (
                tuned_df["present_boost"].astype(float).round(8)
                == float_key(active_config["present_boost"])
            )
        ].copy()

        if subset.empty:
            continue

        summary = {
            "organ": organ,
            "ordinal_alpha": float(active_config["ordinal_alpha"]),
            "present_boost": float(active_config["present_boost"]),
            "n_seeds": int(len(subset)),
            "val_loss_mean": subset["val_loss"].mean(),
            "val_loss_std": subset["val_loss"].std(),
            "val_acc_mean": subset["val_acc"].mean(),
            "val_acc_std": subset["val_acc"].std(),
            "recall_class0_mean": subset["recall_class0"].mean(),
            "recall_class0_std": subset["recall_class0"].std(),
            "recall_class1_mean": subset["recall_class1"].mean(),
            "recall_class1_std": subset["recall_class1"].std(),
            "recall_class2_mean": subset["recall_class2"].mean(),
            "recall_class2_std": subset["recall_class2"].std(),
            "macro_recall_mean": subset["macro_recall"].mean(),
            "macro_recall_std": subset["macro_recall"].std(),
        }
        summary_rows.append(summary)

        print(f"\n{organ}")
        print(
            f"  val_loss: "
            f"{summary['val_loss_mean']:.4f} "
            f"± {summary['val_loss_std']:.4f}"
        )
        print(
            f"  val_acc: "
            f"{summary['val_acc_mean']:.3f} "
            f"± {summary['val_acc_std']:.3f}"
        )
        print(
            f"  recall c0/c1/c2: "
            f"{summary['recall_class0_mean']:.3f} / "
            f"{summary['recall_class1_mean']:.3f} / "
            f"{summary['recall_class2_mean']:.3f}"
        )
        print(
            f"  macro recall: "
            f"{summary['macro_recall_mean']:.3f} "
            f"± {summary['macro_recall_std']:.3f}"
        )

    tuned_summary_df = pd.DataFrame(summary_rows)
    atomic_write_csv(
        tuned_summary_df,
        OUTPUT_DIR / f"multiseed_tuned_{TUNED_CONFIG_SIGNATURE}_summary.csv",
    )

    expected_confirmations = len(BEST_CONFIGS) * len(SEEDS)
    active_confirmations = 0

    for organ, config in BEST_CONFIGS.items():
        active_confirmations += int(
            (
                (tuned_df["organ"] == organ)
                & (
                    tuned_df["ordinal_alpha"].astype(float).round(8)
                    == float_key(config["ordinal_alpha"])
                )
                & (
                    tuned_df["present_boost"].astype(float).round(8)
                    == float_key(config["present_boost"])
                )
            ).sum()
        )

    print(
        f"\nActive tuned confirmations: "
        f"{active_confirmations}/{expected_confirmations}"
    )
else:
    print("No tuned confirmation rows are available yet.")


TUNED MULTI-SEED SUMMARY

kidney
  val_loss: 0.7586 ± 0.1405
  val_acc: 0.709 ± 0.070
  recall c0/c1/c2: 0.627 / 0.686 / 0.836
  macro recall: 0.716 ± 0.074

spleen
  val_loss: 0.9562 ± 0.1207
  val_acc: 0.606 ± 0.021
  recall c0/c1/c2: 0.477 / 0.457 / 0.815
  macro recall: 0.583 ± 0.038

Active tuned confirmations: 10/10


In [21]:
# -----------------------------
# Output inventory
# -----------------------------
print("\nCompleted output inventory:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR))


Completed output inventory:
best_configs.json
efficientnet_b0_gray_backbone.pt
hf_home/hub/CACHEDIR.TAG
hf_home/hub/models--timm--efficientnet_b0.ra_in1k/blobs/d569899762ea9b1384ee07f4af64805cf8caa1c55f9253ebb1080dc40e87a2cd
hf_home/hub/models--timm--efficientnet_b0.ra_in1k/refs/main
hf_home/hub/models--timm--efficientnet_b0.ra_in1k/snapshots/1b5383e5f79cc0f7fc067e372f8f26a5fa73f26a/model.safetensors
models/tuned_confirmation_kidney_seed123_alpha1_boost1.pt
models/tuned_confirmation_kidney_seed2024_alpha1_boost1.pt
models/tuned_confirmation_kidney_seed42_alpha1_boost1.pt
models/tuned_confirmation_kidney_seed555_alpha1_boost1.pt
models/tuned_confirmation_kidney_seed7_alpha1_boost1.pt
models/tuned_confirmation_spleen_seed123_alpha1_boost1.pt
models/tuned_confirmation_spleen_seed2024_alpha1_boost1.pt
models/tuned_confirmation_spleen_seed42_alpha1_boost1.pt
models/tuned_confirmation_spleen_seed555_alpha1_boost1.pt
models/tuned_confirmation_spleen_seed7_alpha1_boost1.pt
multiseed_tuned_kid